In [ ]:
library(Seurat)
#library(SeuratData)
library(SeuratDisk)

load("/mnt/beegfs/userdata/d_papakonstantinou/crc/data/integrated_crc.rda")

In [3]:
ls()

[1] "integrated_crc"

In [4]:
SaveH5Seurat(integrated_crc, filename = "/mnt/beegfs/userdata/d_papakonstantinou/crc/data/integrated_crc.h5Seurat")

Creating h5Seurat file for version 3.1.5.9900

Adding counts for SCT

Adding data for SCT

Adding scale.data for SCT

Adding variable features for SCT

No feature-level metadata found for SCT

Writing out SCTModel.list for SCT

Adding cell embeddings for pca

Adding loadings for pca

No projected loadings for pca

Adding standard deviations for pca

No JackStraw data for pca

Adding cell embeddings for harmony

Adding loadings for harmony

Adding projected loadings for harmony

Adding standard deviations for harmony

No JackStraw data for harmony

Adding cell embeddings for umap

No loadings for umap

No projected loadings for umap

No standard deviations for umap

No JackStraw data for umap



In [5]:
Convert("/mnt/beegfs/userdata/d_papakonstantinou/crc/data/integrated_crc.h5Seurat", dest = "/mnt/beegfs/userdata/d_papakonstantinou/crc/data/integrated_crc.h5ad")

Validating h5Seurat file

Adding scale.data from SCT as X

Adding data from SCT as raw

Transfering meta.data to obs

Adding dimensional reduction information for harmony

Adding feature loadings for harmony

Adding dimensional reduction information for pca

Adding feature loadings for pca

Adding dimensional reduction information for umap

Adding SCT_snn as neighbors



# Primary and Pre-treated Subset

In [ ]:
primary_crc = subset(integrated_crc, subset = tumor_type == "Primary" & timepoint == 'Pre-treatment')
print(1)
DefaultAssay(primary_crc) <- "RNA"
primary_crc[['SCT']] <- NULL
primary_list <- SplitObject(primary_crc, split.by="orig.ident")
print(2)
merged_seurat <- merge(x = primary_list[[1]], y = primary_list[2:length(primary_list)],merge.data = TRUE)
print(3)
merged_seurat <- merged_seurat %>%
    NormalizeData() %>%
    FindVariableFeatures(selection.method = "vst", nfeatures = 2000) %>% 
    ScaleData(return.only.var.genes = TRUE) %>%
    SCTransform(vars.to.regress = c("percent_mt"))
print(4)
merged_seurat <- RunPCA(merged_seurat, assay = "SCT", npcs = 50)
print(5)

primary_crc <- RunHarmony(merged_seurat,group.by.vars = c("orig.ident"),
                          reduction = "pca",
                          assay.use = "SCT",
                          reduction.save = "harmony")

print(5)
primary_crc <- RunUMAP(primary_crc, reduction = "harmony", assay = "SCT", dims = 1:40)
print(6)
primary_crc <- FindNeighbors(object =primary_crc, reduction = "harmony")
print(7)
primary_crc <- FindClusters(primary_crc, resolution = c(0.1,0.2, 0.4, 0.6))
save(primary_crc,"/mnt/beegfs/userdata/d_papakonstantinou/crc/data/primary_crc.rda")